In [ ]:
import mne
from mne.datasets import eegbci

In [ ]:
subject = 1
runs = [3, 4]  # left vs right hand motor imagery runs

raw_fnames = eegbci.load_data(subject, runs)
raw = mne.io.concatenate_raws([mne.io.read_raw_edf(f, preload=True) for f in raw_fnames])
raw_filtered = raw.copy().filter(8., 30., fir_design='firwin')

In [ ]:
print(raw.info)
print(raw.get_data().shape)
print(raw_filtered.info)
print(raw_filtered.get_data().shape)

In [ ]:
raw.plot(duration=5, n_channels=10)
raw_filtered.plot(duration=5, n_channels=10)

In [ ]:
events, event_id = mne.events_from_annotations(raw_filtered)
print(event_id)
print(events[:10])

In [ ]:
event_id = dict(left=2, right=3)

epochs = mne.Epochs(
    raw_filtered,
    events,
    event_id=event_id,
    tmin=0.5,
    tmax=2,
    baseline=None,
    preload=True
)

print(epochs)
print(epochs.get_data().shape)

In [ ]:
print(len(raw_fnames))
print(epochs.get_data().shape)
print(raw_fnames)

In [ ]:

X = epochs.get_data()  
y = epochs.events[:, -1]

print(X.shape)
print(y.shape)

X = epochs.get_data()
y = epochs.events[:, -1]
y = y - 2   # convert 2/3 to 0/1

In [ ]:
from mne.decoding import CSP
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

pipeline = Pipeline([
    ('csp', CSP(n_components=9, log=True)),
    ('svm', SVC())
])

param_grid = {
    'svm__kernel': ['linear', 'rbf'],
    'svm__C': [0.1, 1, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=5)
grid.fit(X, y)

print("Best score:", grid.best_score_)
print("Best params:", grid.best_params_)


In [ ]:
best_model = grid.best_estimator_
scores = cross_val_score(best_model, X, y, cv=5)
print(scores.mean())

In [ ]:
print("runs:", runs)
print("band:", "8-12 Hz")
print("window:", "0.5–2.0 s")
print("best params:", grid.best_params_)
print("best cv score:", grid.best_score_)